In [9]:
# Load environment and suppress warnings
import os
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [10]:
# Import required components and initialize deterministic model
from pydantic import BaseModel, Field
from typing import Literal

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableParallel

model = ChatOpenAI(model="gpt-5-mini", temperature=0)

In [11]:
# Define demo inputs (two tickets to show parallel execution works for any input)
ticket_technical = """
Database timeout errors started after the latest deploy.
Users cannot load the dashboard. Please help ASAP.
"""

ticket_billing = """
I was charged twice for the same invoice this month.
Please refund the duplicate charge and confirm.
"""

In [12]:
# Define schema and build the extraction chain (issue and category)
class TicketExtraction(BaseModel):
    issue: str = Field(description="Short issue phrase, 2 to 6 words")
    category: Literal["technical", "billing", "other"] = Field(description="Issue category")

extract_parser = PydanticOutputParser(pydantic_object=TicketExtraction)

extract_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Extract structured fields from support tickets."),
        (
            "human",
            "Ticket:\n{ticket_text}\n\n"
            "Instructions:\n"
            "- issue: a short phrase (2 to 6 words)\n"
            "- category: technical, billing, or other\n\n"
            "Return the extraction in the required format.\n{format_instructions}",
        ),
    ]
).partial(format_instructions=extract_parser.get_format_instructions())

extract_chain = extract_prompt | model | extract_parser

In [13]:
# Build the priority classification chain with explicit rules
priority_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Classify priority using the rules. Output only High, Medium, or Low."),
        (
            "human",
            "Ticket:\n{ticket_text}\n\n"
            "Rules:\n"
            "- High: production outage, many users blocked, critical security risk, data loss\n"
            "- Medium: important issue but workaround exists, billing dispute/refund, single team impacted\n"
            "- Low: minor question, cosmetic issue, request for information\n\n"
            "Return exactly one label: High, Medium, or Low.",
        ),
    ]
)

priority_chain = priority_prompt | model | StrOutputParser()

In [14]:
# Compose parallel execution (run extraction and priority at the same time)
parallel_chain = RunnableParallel(
    extraction=extract_chain,
    priority=priority_chain,
)

In [15]:
# Run parallel execution for the technical ticket
out_tech = parallel_chain.invoke({"ticket_text": ticket_technical})

print("=== PARALLEL OUTPUT (TECHNICAL) ===")
print("Issue   :", out_tech["extraction"].issue)
print("Category:", out_tech["extraction"].category)
print("Priority:", out_tech["priority"])

=== PARALLEL OUTPUT (TECHNICAL) ===
Issue   : Database timeout errors
Category: technical
Priority: High


In [16]:
# Run parallel execution for the billing ticket
out_bill = parallel_chain.invoke({"ticket_text": ticket_billing})

print("=== PARALLEL OUTPUT (BILLING) ===")
print("Issue   :", out_bill["extraction"].issue)
print("Category:", out_bill["extraction"].category)
print("Priority:", out_bill["priority"])

=== PARALLEL OUTPUT (BILLING) ===
Issue   : Duplicate charge refund request
Category: billing
Priority: Medium
